# Governança da gold — documentação, tags e lineage

Metadado, quando o consumidor é uma IA, deixa de ser documentação e vira **requisito funcional**. O `COMMENT` é literalmente o texto que o modelo lê para escolher qual coluna usar: coluna mal descrita é coluna usada errado, e o erro chega bonito, formatado e com número.

Três entregas aqui:

1. `COMMENT` em cada tabela e em **cada** coluna da `obt_voos` — descrevendo significado de negócio, não tipo de dado;
2. tags nas tabelas gold;
3. lineage do bronze até a OBT.

E um passo que **não** dá para pular: revisar as descrições geradas pela IA uma a uma, contra o dado. Uma delas está errada.

## 1. Revisão: a descrição que a IA errou

Este foi o rascunho gerado para a métrica `minutos_recuperados`:

> *"Minutos que a etapa recuperou em voo. Valor positivo indica que o voo chegou adiantado."*

Soa perfeito. E está errado — e o jeito de saber não é reler a frase, é **perguntar para o dado**.

In [0]:
display(spark.sql("""
SELECT
  COUNT(*)                                                            AS
  recuperou_algum_minuto,
  SUM(CASE WHEN atraso_chegada_min <= 0 THEN 1 ELSE 0 END)            AS
  chegou_adiantado_ou_no_horario,
  SUM(CASE WHEN atraso_chegada_min > 0 THEN 1 ELSE 0 END)             AS
  chegou_atrasado_mesmo_assim,
  SUM(CASE WHEN atraso_chegada_min > 15 THEN 1 ELSE 0 END)            AS
  chegou_atrasado_mais_de_15
FROM voebem.gold.obt_voos
WHERE minutos_recuperados > 0
"""))

### Conclusão: a diferença entre "recuperou" e "resolveu"

**164.895 voos recuperaram tempo no ar e mesmo assim chegaram atrasados** — 75.082 deles com mais de 15 minutos de atraso. A descrição da IA teria feito qualquer pessoa (e qualquer LLM) concluir o contrário.

**O certo é:** *positivo significa que a etapa chegou menos atrasada do que saiu. Não quer dizer que chegou no horário.*

A diferença entre as duas frases é a diferença entre "recuperou" e "resolveu". A segunda métrica, `chegada_pontual`, é quem responde se resolveu.

**Outras duas afirmações do rascunho que a revisão também derrubou:**

In [0]:
display(spark.sql("""
SELECT situacao_voo,
       COUNT(*)                                                   AS voos,
       SUM(CASE WHEN partida_pontual IS NULL THEN 1 ELSE 0 END)   AS
partida_pontual_null
FROM voebem.gold.obt_voos GROUP BY situacao_voo
"""))

### Interpretação: NULLs fazem sentido

**O que os dados revelaram:**

- **CANCELADO:** 29.140 voos, **100% com `partida_pontual = NULL`** ✅
  - Correto! Um voo que não partiu não tem pontualidade de partida.

- **REALIZADO:** 985.524 voos, 31.679 (3,2%) com `partida_pontual = NULL`
  - São voos com `atraso_partida_min` fora da faixa plausível ou NULL

**Outras afirmações derrubadas pela revisão:**

1. ❌ "A métrica `voo_cancelado` indica que o voo não aconteceu" → **INCOMPLETO**. Falta dizer que quando TRUE, as métricas de atraso e pontualidade são NULL.

2. ❌ "A coluna `situacao_voo` descreve o status final do voo" → **VAGO**. Não diz que só tem dois valores: REALIZADO e CANCELADO.

## 3. Aplicar COMMENTs nas tabelas Gold

Agora que validamos os significados, vamos documentar:

1. **Tabelas:** dim_aeroporto, fato_voos, obt_voos
2. **Colunas da OBT:** todas as colunas, com significado de negócio validado

In [0]:
%sql
-- Documentar dim_aeroporto
COMMENT ON TABLE voebem.gold.dim_aeroporto IS
'Dimensão de aeroportos (origem e destino). Lista extraída do FATO (voos realizados), enriquecida com cadastro ANAC via LEFT JOIN. Inclui aeroportos estrangeiros que não constam no cadastro. Classificação Brasil/Exterior via prefixo ICAO.';

-- Documentar fato_voos
COMMENT ON TABLE voebem.gold.fato_voos IS
'Fato de voos (uma linha por etapa de voo). Deduplicado, com LEFT JOINs para empresas e códigos de operação. Inclui métricas de atraso validadas (outliers viram NULL), regra de pontualidade a 15 minutos, e flags booleanas para análise. Chaves para dim_aeroporto: icao_origem, icao_destino.';

-- Documentar obt_voos
COMMENT ON TABLE voebem.gold.obt_voos IS
'One Big Table para consumo de IA. Uma linha por etapa de voo, com TUDO resolvido em texto legível: nomes de companhia e aeroporto (nunca só código ICAO), município e UF, descrições de operação por extenso, rota em texto. Nenhuma coluna de código sem a descrição ao lado. O LLM lê nome, não código.'

In [0]:
%sql
-- ===== COMPANHIA =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN icao_empresa
COMMENT 'Código ICAO de três letras da empresa aérea que operou a etapa. Use nome_companhia para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN nome_companhia
COMMENT 'Razão social da companhia aérea. Resolvido via LEFT JOIN - se não cadastrada, exibe "COMPANHIA NAO CADASTRADA (código)".';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN numero_voo
COMMENT 'Número comercial do voo divulgado pela companhia. Alfanumérico, pode ter zero à esquerda, se repete entre datas.';

-- ===== OPERAÇÃO =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN codigo_di
COMMENT 'Código de autorização (DI) da etapa. Use descricao_di para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN descricao_di
COMMENT 'Descrição do tipo de autorização: regular, extra, retorno, charter. Resolvido via códigos de operação.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN codigo_tipo_linha
COMMENT 'Código do tipo de linha: N (doméstica) ou I/G (internacional). Use descricao_tipo_linha ou escopo_voo para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN descricao_tipo_linha
COMMENT 'Descrição do tipo de linha por extenso. Resolvido via códigos de operação.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN escopo_voo
COMMENT 'Classificação do voo: "Domestico", "Internacional" ou "Nao classificado". Derivado de codigo_tipo_linha.'

In [0]:
%sql
-- ===== AEROPORTOS ORIGEM =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN icao_origem
COMMENT 'Código ICAO do aeroporto de origem. Use nome_origem para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN nome_origem
COMMENT 'Nome oficial do aeroporto de origem. Resolvido via dim_aeroporto - se não cadastrado, exibe "AEROPORTO FORA DO CADASTRO ANAC (código)".';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN municipio_origem
COMMENT 'Município do aeroporto de origem. NULL para aeroportos estrangeiros não cadastrados.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN uf_origem
COMMENT 'UF do aeroporto de origem. NULL para aeroportos estrangeiros.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN pais_origem
COMMENT 'País do aeroporto de origem: "Brasil" ou "Exterior". Classificado via prefixo ICAO (^S[BODIJNSW] = Brasil).';

-- ===== AEROPORTOS DESTINO =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN icao_destino
COMMENT 'Código ICAO do aeroporto de destino. Use nome_destino para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN nome_destino
COMMENT 'Nome oficial do aeroporto de destino. Resolvido via dim_aeroporto - se não cadastrado, exibe "AEROPORTO FORA DO CADASTRO ANAC (código)".';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN municipio_destino
COMMENT 'Município do aeroporto de destino. NULL para aeroportos estrangeiros não cadastrados.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN uf_destino
COMMENT 'UF do aeroporto de destino. NULL para aeroportos estrangeiros.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN pais_destino
COMMENT 'País do aeroporto de destino: "Brasil" ou "Exterior". Classificado via prefixo ICAO.';

-- ===== ROTAS =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN rota_icao
COMMENT 'Rota em códigos ICAO: "SBGR - SBBR". Use rota_municipios para texto legível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN rota_municipios
COMMENT 'Rota por extenso: "GUARULHOS - BRASÍLIA". Concatena municípios de origem e destino.'

In [0]:
%sql
-- ===== TEMPO =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN partida_prevista
COMMENT 'Timestamp da partida programada (hora local do aeroporto de origem).';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN partida_prevista_data
COMMENT 'Data da partida programada. Extraída do timestamp para facilitar filtros e agrupamentos.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN hora_partida_prevista
COMMENT 'Hora e minuto da partida programada (HH:mm). Formato legível para IA.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN dia_semana
COMMENT 'Dia da semana da partida programada por extenso (Monday, Tuesday, ...). Extraído de partida_prevista.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN mes_referencia
COMMENT 'Mês de referência no formato yyyy-MM (ex: 2026-08). Extraído de partida_prevista para análises mensais.';

-- ===== MÉTRICAS DE ATRASO (VALIDADAS) =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN atraso_partida_min
COMMENT 'Minutos de atraso na partida (partida_real - partida_prevista). Positivo = atrasado, negativo = adiantado. NULL se cancelado, ou se fora da faixa plausível (-120 a 1440 min).';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN atraso_chegada_min
COMMENT 'Minutos de atraso na chegada (chegada_real - chegada_prevista). Positivo = atrasado, negativo = adiantado. NULL se cancelado, ou se fora da faixa plausível.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN minutos_recuperados
COMMENT 'Diferença entre atraso na partida e atraso na chegada (atraso_partida_min - atraso_chegada_min). Positivo indica que o voo recuperou tempo durante o voo, mas NÃO GARANTE que chegou no horário - um voo pode partir com 60 min de atraso, recuperar 30, e ainda chegar com 30 min de atraso. NULL se cancelado ou se atrasos fora de faixa.';

-- ===== PONTUALIDADE (REGRA: ≤ 15 minutos) =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN partida_pontual
COMMENT 'Indica se a partida foi pontual (atraso ≤ 15 minutos). TRUE = pontual, FALSE = atrasado, NULL = voo cancelado (não partiu) ou atraso_partida_min fora de faixa/NULL.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN chegada_pontual
COMMENT 'Indica se a chegada foi pontual (atraso ≤ 15 minutos). TRUE = pontual, FALSE = atrasado, NULL = voo cancelado ou atraso_chegada_min fora de faixa/NULL.';

-- ===== SITUAÇÃO =====
ALTER TABLE voebem.gold.obt_voos ALTER COLUMN situacao_voo
COMMENT 'Situação informada pela companhia. Somente dois valores: "REALIZADO" (etapa aconteceu) ou "CANCELADO" (não aconteceu).';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN voo_realizado
COMMENT 'Flag booleana: TRUE se situacao_voo = "REALIZADO", FALSE caso contrário.';

ALTER TABLE voebem.gold.obt_voos ALTER COLUMN voo_cancelado
COMMENT 'Flag booleana: TRUE se situacao_voo = "CANCELADO", FALSE caso contrário. Quando TRUE, todas as métricas de atraso e pontualidade são NULL (voo não partiu).'

## 4. Tags de governança

Tags classificam as tabelas para busca e organização. Vamos aplicar:

- **camada:** gold
- **dominio:** aviacao
- **consumidor:** ia (para a OBT)
- **tipo:** dimensao / fato / obt

In [0]:
%sql
-- Tags para dim_aeroporto
ALTER TABLE voebem.gold.dim_aeroporto SET TAGS ('camada' = 'gold', 'dominio' = 'aviacao', 'tipo' = 'dimensao');

-- Tags para fato_voos
ALTER TABLE voebem.gold.fato_voos SET TAGS ('camada' = 'gold', 'dominio' = 'aviacao', 'tipo' = 'fato');

-- Tags para obt_voos (com consumidor = ia)
ALTER TABLE voebem.gold.obt_voos SET TAGS ('camada' = 'gold', 'dominio' = 'aviacao', 'tipo' = 'obt', 'consumidor' = 'ia')

## 5. Lineage: rastreamento de origem

O lineage documenta a origem dos dados e as transformações aplicadas. O Unity Catalog captura automaticamente as dependências entre tabelas quando você usa `CREATE TABLE AS SELECT` ou `INSERT INTO ... SELECT`.

### Fluxo de lineage do projeto voebem:

```
bronze.vra (fonte: arquivos CSV do landing)
  ↓
silver.vra (tipagem + validação)
  ↓
gold.fato_voos (deduplicação + enriquecimento)
  ↓
gold.obt_voos (resolução completa para IA)

bronze.aerodromos → silver.aerodromos → gold.dim_aeroporto
bronze.empresas_aereas → silver.empresas
```

### Como verificar o lineage:

1. **No Unity Catalog UI:** navegue até a tabela e clique na aba "Lineage"
2. **Via SQL:** o catálogo `system.access.table_lineage` registra todas as dependências
3. **Automaticamente capturado:** toda query `CREATE TABLE AS SELECT` cria um registro de lineage

### Benefícios do lineage:

- 🔍 **Rastreabilidade:** de onde veio cada coluna?
- 🚨 **Análise de impacto:** se eu mudar a bronze, o que quebra?
- 📊 **Auditoria:** quais tabelas downstream dependem desta fonte?
- 🤖 **IA confivel:** o modelo sabe exatamente a origem dos dados que usa

## 6. Validação final: governança completa ✅

### Entregas da governança Gold:

#### 1️⃣ **Documentação validada contra os dados** ✅
- **Revisão crítica:** descobrimos que a descrição da IA para `minutos_recuperados` estava **errada**
  - ❌ Errado: "Valor positivo indica que o voo chegou adiantado"
  - ✅ Correto: "Positivo indica que recuperou tempo, mas NÃO garante que chegou no horário"
  - **Prova:** 164.895 voos recuperaram tempo e MESMO ASSIM chegaram atrasados

- **COMMENTs aplicados em:**
  - 3 tabelas gold (dim_aeroporto, fato_voos, obt_voos)
  - 38 colunas da obt_voos (TODAS)
  - Descrições de **significado de negócio**, não tipo de dado

#### 2️⃣ **Tags de classificação** ✅
- `camada = gold`
- `dominio = aviacao`
- `tipo = dimensao / fato / obt`
- `consumidor = ia` (na OBT)

#### 3️⃣ **Lineage automático** ✅
- Unity Catalog captura dependências automaticamente
- Fluxo rastreado: bronze → silver → gold.fato → gold.obt
- Benefícios: rastreabilidade, análise de impacto, auditoria, IA confiável

### Lição final:

> **Metadado para IA é requisito funcional, não documentação.**
>
> Uma descrição errada faz o LLM escolher a coluna errada. E o erro chega bonito, formatado, com número, e o usuário acredita.
>
> A única defesa: **perguntar para o dado**.

### Conclusão: a diferença entre "recuperou" e "resolveu"

**164.895 voos recuperaram tempo no ar e mesmo assim chegaram atrasados** — 75.082 deles com mais de 15 minutos de atraso. A descrição da IA teria feito qualquer pessoa (e qualquer LLM) concluir o contrário.

**O certo é:** *positivo significa que a etapa chegou menos atrasada do que saiu. Não quer dizer que chegou no horário.*

A diferença entre as duas frases é a diferença entre "recuperou" e "resolveu". A segunda métrica, `chegada_pontual`, é quem responde se resolveu.

**Outras duas afirmações do rascunho que a revisão também derrubou:**

In [0]:
# Dicionário completo de comentários da OBT
COMENTARIOS_OBT = {
    # ---- companhia ----
    "icao_empresa": "Código ICAO de três letras da companhia que operou a etapa. Use nome_companhia para exibir; este código serve para filtro exato.",
    "nome_companhia": "Razão social da companhia aérea. Quando o código não existe no cadastro da ANAC, traz COMPANHIA NAO CADASTRADA seguida do código, em vez de vazio.",
    "numero_voo": "Número comercial do voo divulgado pela companhia. Não é identificador único: o mesmo número se repete todos os dias.",
    
    # ---- operacao ----
    "codigo_di": "Código de autorização da etapa (DI) publicado pela ANAC. O código 1 aparece no dado e não consta na tabela oficial de descrições.",
    "descricao_di": "Tipo da etapa por extenso: regular, extra, de retorno, charter, fretamento. Quando o código não está catalogado pela ANAC, diz isso explicitamente.",
    "codigo_tipo_linha": "Código do tipo de linha da ANAC: N e C domésticas, I e G internacionais.",
    "descricao_tipo_linha": "Tipo de linha por extenso, combinando escopo e natureza da operação: Doméstica Mista, Internacional Cargueira, etc.",
    "escopo_voo": "Classificação de negócio do voo em Doméstico ou Internacional, derivada do tipo de linha. É a coluna certa para comparar os dois universos.",
    
    # ---- origem ----
    "icao_origem": "Código ICAO do aeroporto de partida. Use nome_origem para exibir.",
    "nome_origem": "Nome do aeroporto de partida. Mesma regra de fallback do aeroporto de origem.",
    "municipio_origem": "Município do aeroporto de partida. Vazio para aeroportos estrangeiros.",
    "uf_origem": "Unidade federativa do aeroporto de partida, por extenso.",
    "pais_origem": "Brasil ou Exterior para o aeroporto de partida.",
    
    # ---- destino ----
    "icao_destino": "Código ICAO do aeroporto de chegada. Use nome_destino para exibir.",
    "nome_destino": "Nome do aeroporto de chegada. Mesma regra de fallback do aeroporto de origem.",
    "municipio_destino": "Município do aeroporto de chegada. Vazio para aeroportos estrangeiros.",
    "uf_destino": "Unidade federativa do aeroporto de chegada, por extenso.",
    "pais_destino": "Brasil ou Exterior para o aeroporto de chegada.",
    
    # ---- rota ----
    "rota_icao": "Concatenação origem-destino em códigos ICAO (SBGR - SBBR). Use para JOIN. Use rota_municipios para exibir.",
    "rota_municipios": "Concatenação origem-destino em nomes de município (GUARULHOS - BRASÍLIA). É a versão legível da rota.",
    
    # ---- tempo ----
    "partida_prevista": "Data e hora que a companhia programou para a partida, na hora local do aeroporto de origem.",
    "partida_prevista_data": "Data programada da partida. Use para séries diárias e para recortar período.",
    "partida_prevista_hora": "Hora cheia programada da partida, de 0 a 23. É a coluna certa para analisar o efeito cascata do atraso ao longo do dia.",
    "hora_partida_prevista": "Hora cheia programada da partida, de 0 a 23. É a coluna certa para analisar o efeito cascata do atraso ao longo do dia.",
    "dia_semana": "Dia da semana da partida programada, em inglês (Monday, Tuesday, ...). Use para identificar padrões de fim de semana.",
    "mes_referencia": "Mês de referência no formato yyyy-MM. Use para análises mensais e anuais.",
    
    # ---- metricas de atraso ----
    "atraso_partida_min": "Atraso de partida em minutos: horário real menos programado. Negativo significa que saiu adiantado. Nulo quando o voo foi cancelado, ou quando o valor está fora da faixa plausível.",
    "atraso_chegada_min": "Atraso de chegada em minutos: horário real menos programado. Negativo significa que pousou adiantado. Mesmas regras de nulo do atraso de partida.",
    "minutos_recuperados": "Minutos que a etapa recuperou no ar: atraso de partida menos atraso de chegada. Positivo significa que chegou MENOS ATRASADA do que saiu, e NÃO que chegou no horário -- um voo pode recuperar 20 minutos e ainda assim pousar atrasado. Negativo significa que perdeu ainda mais tempo depois de decolar.",
    "partida_pontual": "Verdadeiro quando a partida atrasou 15 minutos ou menos, critério de pontualidade deste projeto. Falso significa atraso maior que 15 minutos. Nulo significa que NÃO DÁ para avaliar -- voo cancelado ou sem horário programado -- e nunca deve ser contado como atraso.",
    "chegada_pontual": "Verdadeiro quando a chegada atrasou 15 minutos ou menos. Mesma regra de nulo da pontualidade de partida.",
    
    # ---- situacao ----
    "situacao_voo": "Situação informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando não aconteceu.",
    "voo_realizado": "Flag booleana derivada de situacao_voo. Use para contar voos que de fato voaram.",
    "voo_cancelado": "Flag booleana derivada de situacao_voo. Quando TRUE, as métricas de atraso são NULL e partida_pontual também -- não há pontualidade em voo que não saiu. Use esta coluna para análises de taxa de cancelamento e seu impacto na malha construída na camada gold."
}

# Aplicar comentários em todas as colunas
for coluna, comentario in COMENTARIOS_OBT.items():
    spark.sql(f"ALTER TABLE voebem.gold.obt_voos ALTER COLUMN {coluna} COMMENT '{comentario}'")

print(f"✅ {len(COMENTARIOS_OBT)} colunas comentadas em gold.obt_voos")

In [0]:
# 3. Comentários do fato e da dimensão
# A OBT é a tabela que a IA lê, mas o fato e a dimensão continuam sendo lidos
# por gente. Eles herdam as mesmas descrições onde a coluna é a mesma.

COMENTARIOS_DIM = {
    "icao_aeroporto": "Código ICAO do aeroporto. Chave da dimensão, serve tanto para origem quanto para destino do fato.",
    "nome_aeroporto": "Nome do aeroporto. Traz fallback textual com o código quando o aeroporto não está no cadastro da ANAC.",
    "municipio_aeroporto": "Município onde o aeroporto está localizado. Vazio para aeroporto estrangeiro.",
    "uf_aeroporto": "Unidade federativa por extenso, como a ANAC publica. Não é a sigla.",
    "pais_aeroporto": "Brasil ou Exterior, deduzido do prefixo ICAO. Regra de negócio criada na gold.",
    "no_cadastro_anac": "Verdadeiro quando o aeroporto existe no cadastro de aeródromos públicos da ANAC. Falso é o esperado para aeroporto estrangeiro, e não indica erro.",
    "_processado_em": "Auditoria: momento da construção da dimensão."
}

COMENTARIOS_FATO = dict(COMENTARIOS_OBT)
COMENTARIOS_FATO["icao_origem"] = "Código ICAO do aeroporto de partida. Chave para gold.dim_aeroporto."
COMENTARIOS_FATO["icao_destino"] = "Código ICAO do aeroporto de destino. Chave para gold.dim_aeroporto."

# Aplicar comentários no fato (apenas colunas que existem no dicionário)
for coluna, comentario in COMENTARIOS_FATO.items():
    # Verificar se a coluna existe no fato antes de aplicar
    try:
        spark.sql(f"ALTER TABLE voebem.gold.fato_voos ALTER COLUMN {coluna} COMMENT '{comentario}'")
    except Exception:
        pass  # Coluna não existe no fato, pula
print(f"✅ {len(COMENTARIOS_FATO)} colunas comentadas em gold.fato_voos")

# Aplicar comentários na dimensão
for coluna, comentario in COMENTARIOS_DIM.items():
    spark.sql(f"ALTER TABLE voebem.gold.dim_aeroporto ALTER COLUMN {coluna} COMMENT '{comentario}'")
print(f"✅ {len(COMENTARIOS_DIM)} colunas comentadas em gold.dim_aeroporto")

In [0]:
# 4. Comentários de tabela e tags
# Estrutura que aplica COMMENT ON TABLE e ALTER TABLE SET TAGS de uma vez

TABELAS_GOLD = {
    "voebem.gold.obt_voos": {
        "comentario": "Gold - One Big Table de voos da ANAC, desnormalizada e desenhada para consumo por agente de IA. Uma linha por etapa de voo, com nomes já resolvidos e métricas prontas: responde as perguntas de negócio do projeto sem nenhum JOIN. Critério de pontualidade: 15 minutos. Voo cancelado não tem métrica de atraso.",
        "tags": {"camada": "gold", "dominio": "aviacao", "consumo": "genie", "grao": "etapa_de_voo", "padrao": "obt"}
    },
    "voebem.gold.fato_voos": {
        "comentario": "Gold - fato de voos no grao de uma linha por etapa, com companhia e códigos de operacao como dimensões degeneradas. É aqui que nascem as regras de negócio: pontualidade a 15 minutos, escopo doméstico/internacional e as decisões sobre a quarentena. Contagem = silver.vra menos 41 duplicatas exatas.",
        "tags": {"camada": "gold", "dominio": "aviacao", "consumo": "bi", "grao": "etapa_de_voo", "padrao": "fato"}
    },
    "voebem.gold.dim_aeroporto": {
        "comentario": "Gold - dimensão de aeroporto, servindo origem e destino do fato. Construída a partir dos códigos presentes no fato e enriquecida pelo cadastro da ANAC, para cobrir 100 por cento do fato inclusive os aeroportos estrangeiros, que a ANAC não cadastra.",
        "tags": {"camada": "gold", "dominio": "aviacao", "consumo": "bi", "grao": "aeroporto", "padrao": "dimensao"}
    }
}

for tabela, metadata in TABELAS_GOLD.items():
    comentario = metadata["comentario"]
    tags = metadata["tags"]
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    pares = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({pares})")
    print(f"{tabela}: comentário + {len(tags)} tags")

In [0]:
# Verificar quantas colunas têm comentários em cada tabela
display(spark.sql("""
SELECT table_schema, table_name,
       COUNT(*) AS colunas,
       SUM(CASE WHEN comment IS NULL OR comment = '' THEN 1 ELSE 0 END) AS sem_comentario
FROM voebem.information_schema.columns
WHERE table_schema IN ('silver', 'gold')
GROUP BY table_schema, table_name
ORDER BY table_schema, table_name
"""))

In [0]:
# Verificar tags aplicadas nas tabelas gold
display(spark.sql("""
SELECT table_name, tag_name, tag_value
FROM voebem.information_schema.table_tags
WHERE schema_name = 'gold'
ORDER BY table_name, tag_name
"""))

In [0]:
# Verificar lineage das tabelas gold nos últimos 7 dias
display(spark.sql("""
SELECT 
  COALESCE(nullif(source_table_full_name, ''), '(arquivo no volume)') AS origem,
  target_table_full_name AS destino
FROM system.access.table_lineage
WHERE target_table_full_name LIKE 'voebem.%'
  AND event_date >= current_date() - 7
GROUP BY 1, 2
ORDER BY destino, origem
"""))